# Galaxy morphology classification with AstroLens GCNN

A tutorial-scale training example for `astrolens.models.gcnn.GCNN`
(Pandya et al., 2023, https://arxiv.org/abs/2311.01500), a group-equivariant
CNN, using [`UniverseTBD/mmu_gz10`](https://huggingface.co/datasets/UniverseTBD/mmu_gz10) —
a MultimodalUniverse-formatted copy of **Galaxy10 DECals** (17,736 galaxies,
10 discrete morphology classes), the same dataset the paper's own
[reference implementation](https://github.com/snehjp2/GCNNMorphology) trains on,
split 70/10/20 train/val/test.

Optimizer, schedule, and augmentation follow the reference implementation's
`D8.yaml` / `train.py`: `AdamW` (`lr=1e-2`, `weight_decay=1e-4`), `MultiStepLR`
decaying by `0.1`, and the same rotation/affine/flip augmentation. Model,
batch size, and epoch count are set in the cells below. To reproduce the
paper's reported results directly, follow the reference implementation and
its `src/config/*.yaml` files.

## Install example-only dependencies

Not part of AstroLens' core install (`requirements.txt`) — only needed for this example.

In [1]:
!pip install -q datasets torchvision scikit-learn

## Imports

In [2]:
import io

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
from sklearn.model_selection import train_test_split

import astrolens

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Load the dataset and split 70/10/20

Galaxy10 DECals' standard 10 classes (astroNN convention): disturbed, merging,
round smooth, in-between round smooth, cigar-shaped smooth, barred spiral,
unbarred tight spiral, unbarred loose spiral, edge-on without bulge, edge-on
with bulge.

In [ ]:
IMG_SIZE = 255  # GCNN.img_size: sizes the MaskModule's inscribed-circle mask
BATCH_SIZE = 32  # reference uses 128; doesn't fit at 255x255 with group-equivariant conv here
CLASS_NAMES = [
    "disturbed",
    "merging",
    "round_smooth",
    "in_between_round_smooth",
    "cigar_shaped_smooth",
    "barred_spiral",
    "unbarred_tight_spiral",
    "unbarred_loose_spiral",
    "edge_on_no_bulge",
    "edge_on_with_bulge",
]
NUM_CLASSES = len(CLASS_NAMES)
NUM_WORKERS = 4

gz10 = load_dataset("UniverseTBD/mmu_gz10", split="train")
labels = gz10["gz10_label"]

train_idx, rest_idx = train_test_split(
    range(len(gz10)), train_size=0.7, stratify=labels, random_state=0
)
val_idx, test_idx = train_test_split(
    rest_idx,
    train_size=1 / 3,  # 1/3 of the remaining 30% -> 10% val, 20% test
    stratify=[labels[i] for i in rest_idx],
    random_state=0,
)

# reference implementation's augmentation and normalization
# (src/scripts/train.py): heavy rotation/flip augmentation exercises the
# model's rotation/reflection equivariance during training.
train_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.RandomRotation(180),
        transforms.Resize(IMG_SIZE),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize(IMG_SIZE),
        transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ]
)


class GZ10Dataset(Dataset):
    """Map-style wrapper around an index subset of the HF split, applying transform lazily."""

    def __init__(self, hf_split, indices, transform):
        self.hf_split = hf_split
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        example = self.hf_split[self.indices[i]]
        image = Image.open(io.BytesIO(example["rgb_image"]["bytes"])).convert("RGB")
        return self.transform(image), example["gz10_label"]


train_dataset = GZ10Dataset(gz10, train_idx, train_transform)
val_dataset = GZ10Dataset(gz10, val_idx, eval_transform)
test_dataset = GZ10Dataset(gz10, test_idx, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

len(train_dataset), len(val_dataset), len(test_dataset)

Resolving data files:   0%|          | 0/921 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/766 [00:00<?, ?it/s]

(12415, 1773, 3548)

## Create the model

`gcnn_d4`: dihedral group `D4` (4 rotations x reflection, group order 8).
The reference implementation's own config is `D8` (order 16, `D8.yaml`);
`D4` is used here to keep a full 100-epoch run practical on this GPU
(D8 runs ~2.7x slower per step at this resolution).

In [4]:
model = astrolens.create_model(
    "gcnn_d4",
    img_size=IMG_SIZE,
    in_chans=3,
    num_classes=NUM_CLASSES,
).to(device)

sum(p.numel() for p in model.parameters())

4598030

## Train

`AdamW`, `MultiStepLR` decaying by `0.1`, and class-weighted
cross-entropy (inverse-frequency weights from the train split, following the
same approach as the Linformer example) to counter GZ10's class imbalance —
the reference implementation itself uses unweighted cross-entropy.
Milestones are rescaled from the reference's `[25, 50, 75]` at 100 epochs to
the same 25/50/75% points of this notebook's shorter run.

In [5]:
MAX_EPOCHS = 10
LR = 1e-2
WEIGHT_DECAY = 1e-4
MILESTONES = [round(MAX_EPOCHS * f) for f in (0.25, 0.5, 0.75)]  # reference: [25, 50, 75] at 100 epochs
GAMMA = 0.1

# inverse-frequency class weights from the train split, following the same
# approach as the Linformer example, to counter GZ10's class imbalance
train_counts = torch.bincount(
    torch.tensor([labels[i] for i in train_idx]), minlength=NUM_CLASSES
).float()
class_weights = (train_counts.sum() / (NUM_CLASSES * train_counts)).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=MILESTONES, gamma=GAMMA)


def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            count += images.size(0)

    return total_loss / count, correct / count


for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(
        f"epoch {epoch}/{MAX_EPOCHS} "
        f"train_loss={train_loss:.3f} train_acc={train_acc:.3f} "
        f"val_loss={val_loss:.3f} val_acc={val_acc:.3f} "
        f"lr={scheduler.get_last_lr()[0]:.2e}"
    )

epoch 1/10 train_loss=2.059 train_acc=0.221 val_loss=2.070 val_acc=0.238 lr=1.00e-02


epoch 2/10 train_loss=1.740 train_acc=0.334 val_loss=1.661 val_acc=0.390 lr=1.00e-03


epoch 3/10 train_loss=1.492 train_acc=0.443 val_loss=1.419 val_acc=0.473 lr=1.00e-03


epoch 4/10 train_loss=1.422 train_acc=0.470 val_loss=1.390 val_acc=0.497 lr=1.00e-03


epoch 5/10 train_loss=1.373 train_acc=0.488 val_loss=1.369 val_acc=0.503 lr=1.00e-04


epoch 6/10 train_loss=1.309 train_acc=0.514 val_loss=1.247 val_acc=0.549 lr=1.00e-04


epoch 7/10 train_loss=1.294 train_acc=0.517 val_loss=1.243 val_acc=0.541 lr=1.00e-04


epoch 8/10 train_loss=1.286 train_acc=0.522 val_loss=1.234 val_acc=0.552 lr=1.00e-05


epoch 9/10 train_loss=1.263 train_acc=0.536 val_loss=1.228 val_acc=0.556 lr=1.00e-05


epoch 10/10 train_loss=1.263 train_acc=0.532 val_loss=1.234 val_acc=0.550 lr=1.00e-05


## Evaluate on the held-out test split

In [6]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())
        y_true.extend(labels.tolist())

test_acc = accuracy_score(y_true, y_pred)
test_f1_macro = f1_score(y_true, y_pred, average="macro")
print(f"test_acc={test_acc:.3f} test_f1_macro={test_f1_macro:.3f}\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

test_acc=0.548 test_f1_macro=0.516

                         precision    recall  f1-score   support

              disturbed       0.31      0.38      0.34       216
                merging       0.66      0.36      0.47       371
           round_smooth       0.71      0.81      0.76       529
in_between_round_smooth       0.57      0.69      0.62       405
    cigar_shaped_smooth       0.20      0.72      0.31        67
          barred_spiral       0.42      0.23      0.30       409
  unbarred_tight_spiral       0.44      0.49      0.47       366
  unbarred_loose_spiral       0.49      0.37      0.42       525
       edge_on_no_bulge       0.70      0.79      0.74       285
     edge_on_with_bulge       0.72      0.74      0.73       375

               accuracy                           0.55      3548
              macro avg       0.52      0.56      0.52      3548
           weighted avg       0.56      0.55      0.54      3548



## Save the trained weights

Saved for reuse by `examples/gz10_gcnn_analysis.ipynb` (one-pixel attack and latent-space analysis).

In [ ]:
CHECKPOINT_PATH = "gcnn_d4.pt"
torch.save(model.state_dict(), CHECKPOINT_PATH)
CHECKPOINT_PATH